# Ford GoBike: different duration patterns, shared weekday peaks
## Part II — Explanatory analysis
### Mohamed Feras Elghandour

**Theme:** How do subscriber and customer trips differ in duration and timing, and what does weekday/weekend context add?

**Audience:** Bike-share operations and product teams considering when demand occurs and how long bikes are occupied.

**Context and source:** 183,412 recorded trips in the San Francisco Bay Area during February 2019. [Udacity-provided Ford GoBike data](https://video.udacity-data.com/topher/2020/October/5f91cf38_201902-fordgobike-tripdata/201902-fordgobike-tripdata.csv). Subscriber/customer describes the recorded trip category, not a unique person. The analysis retains all trips; missing demographic fields do not affect the selected features.

The four findings below come directly from Part I. Counts represent observed trip volume, medians summarize skewed durations, and normalized hour profiles compare timing independently of group size.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 120,
                     'font.size': 11, 'axes.titlesize': 14})
USER_ORDER = ['Subscriber', 'Customer']
DAY_ORDER = ['Weekday', 'Weekend']
COLORS = {'Subscriber': '#0072B2', 'Customer': '#D55E00'}
MARKERS = {'Subscriber': 'o', 'Customer': 's'}

def finish(ax, title, xlabel, ylabel):
    """Apply consistent titles, units, and uncluttered chart formatting."""
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    sns.despine(ax=ax)
    ax.figure.tight_layout()

def log_minutes(ax):
    """Keep original minute labels on a logarithmic duration axis."""
    ax.set_yscale('log')
    ax.set_yticks([1, 2, 5, 10, 20, 60, 180, 600, 1440])
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_ylim(1, 1600)

def haversine_km(frame):
    """Calculate great-circle distance between recorded station coordinates."""
    lat1, lat2, lon1, lon2 = map(np.radians, [
        frame['start_station_latitude'], frame['end_station_latitude'],
        frame['start_station_longitude'], frame['end_station_longitude']])
    term = (np.sin((lat2 - lat1) / 2)**2
            + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2)
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(term, 0, 1)))

trips = pd.read_csv(Path('201902-fordgobike-tripdata.csv'))
original_shape = trips.shape
trips['start_time'] = pd.to_datetime(trips['start_time'], errors='raise')
trips['end_time'] = pd.to_datetime(trips['end_time'], errors='raise')
trips['duration_min'] = trips['duration_sec'] / 60
trips['start_hour'] = trips['start_time'].dt.hour
trips['weekday'] = trips['start_time'].dt.dayofweek
trips['day_type'] = np.where(trips['weekday'] < 5, 'Weekday', 'Weekend')
trips['distance_km'] = haversine_km(trips)
trips['user_type'] = pd.Categorical(trips['user_type'], USER_ORDER)
trips['day_type'] = pd.Categorical(trips['day_type'], DAY_ORDER)
# Count calendar exposure, rather than treating the two day groups as equal.
calendar = trips[['start_time']].assign(date=trips['start_time'].dt.normalize())
calendar = calendar.drop_duplicates('date')
day_exposure = calendar['date'].dt.dayofweek.map(
    lambda day: 'Weekday' if day < 5 else 'Weekend').value_counts()

## 1. Which group accounts for most trips?
Subscriber trips dominate volume, so the timing and duration needs of this group matter strongly for overall operations.

In [ ]:
counts = trips['user_type'].value_counts().reindex(USER_ORDER)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(USER_ORDER, counts, color=[COLORS[x] for x in USER_ORDER], height=.55)
for bar, count in zip(bars, counts):
    ax.text(count + 2500, bar.get_y()+bar.get_height()/2,
            f'{count:,} trips ({count/len(trips):.1%})', va='center')
ax.set_xlim(0, 205000)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
finish(ax, 'Subscribers account for 89% of recorded trips', 'Number of trips — February 2019', 'Recorded user type')
plt.show()

**Finding:** 163,544 subscriber trips versus 19,868 customer trips. These are trip counts, not counts of members.

## 2. Which group occupies a bike longer per trip?
Median duration avoids letting the long right tail dominate a comparison of typical trips.

In [ ]:
summary = trips.groupby('user_type', observed=True)['duration_min'].agg(['count', 'median']).reindex(USER_ORDER)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(USER_ORDER, summary['median'], color=[COLORS[x] for x in USER_ORDER], width=.55)
for bar, value in zip(bars, summary['median']):
    ax.text(bar.get_x()+bar.get_width()/2, value+.3, f'{value:.2f} min', ha='center', fontsize=12)
ax.set_ylim(0, 16)
finish(ax, 'Customer median trips last about 5 minutes longer', 'Recorded user type', 'Median trip duration (minutes)')
display(summary.rename(columns={'count': 'Trips', 'median': 'Median duration (min)'}).round(2))
plt.show()

**Finding:** Customer median duration is 13.20 minutes, compared with 8.17 for subscribers, a 5.03-minute difference. All trips are included. This is a descriptive difference, not an effect caused by membership.

## 3. Does the duration difference persist within day types?
Grouping by weekday/weekend tests whether the pooled user-type pattern hides a change in context.

In [ ]:
medians = trips.groupby(['day_type', 'user_type'], observed=True)['duration_min'].median().unstack().reindex(index=DAY_ORDER, columns=USER_ORDER)
fig, ax = plt.subplots(figsize=(10, 5))
medians.plot.bar(ax=ax, rot=0, color=[COLORS[x] for x in USER_ORDER], width=.65)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=4)
ax.set_ylim(0, 19)
ax.legend(title='Recorded user type', loc='upper left')
finish(ax, 'Customer median duration rises on weekends; subscriber duration is stable', 'Day type', 'Median trip duration (minutes)')
display(medians.round(2))
plt.show()

**Finding:** Customer medians increase from 12.40 to 16.05 minutes on weekends, while subscriber medians remain near 8.1 minutes. The user-type gap is present within each day type. These medians use all trips and do not require calendar-day normalization.

## 4. When does demand occur within each group?
Within-group percentages reveal timing patterns without confusing the much larger subscriber group with a different distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
for ax, day_type in zip(axes, DAY_ORDER):
    subset = trips.loc[trips['day_type'] == day_type]
    table = pd.crosstab(subset['start_hour'], subset['user_type']).reindex(index=range(24), columns=USER_ORDER, fill_value=0)
    shares = table.div(table.sum(axis=0), axis=1) * 100
    for user in USER_ORDER:
        ax.plot(shares.index, shares[user], color=COLORS[user], marker=MARKERS[user],
                markersize=4, linestyle='-' if user == 'Subscriber' else '--', label=user)
    ax.set(title=day_type, xlabel='Departure hour (0–23)', xticks=range(0, 24, 3), ylim=(0, 15))
axes[0].set_ylabel('Share of user/day-type trips (%)')
axes[1].legend(title='Recorded user type')
fig.suptitle('Day type changes departure timing in both user groups', fontsize=15)
sns.despine(fig=fig)
fig.tight_layout()
plt.show()

**Finding:** Both groups show weekday peaks near 08:00 and 17:00. Weekend profiles spread across midday and afternoon. Each line sums to 100% within its user/day-type group; the two panels share scales. These are departure timing patterns, not measured reasons for travel.

## Implications and limits
Plan investigations of capacity around shared weekday demand peaks. Customer trips generally keep bikes occupied longer, particularly on weekends; inspect station-level availability before choosing a specific redistribution strategy. Subscriber trip volume remains much larger, including on weekends.

These findings are observational and restricted to February 2019. They do not identify unique riders, prove trip purpose, estimate actual routes, or predict seasonal demand. Validate with additional months and station-level availability data before changing operations.

## Presentation design
Four visuals follow one story: trip volume, typical duration, day-type context, then departure timing. Zero-based bars and numeric labels make magnitudes explicit. Blue/orange consistently identify user types; timing lines also use distinct marker shapes and line styles. Shared facet scales, direct units and within-group percentage labels make comparisons interpretable. Data tables and accompanying text provide the key values independently of chart colors.